<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Pip grabs whatever versions of NumPy, seaborn, and PyPortfolioOpt are current the day you run this, so the notebook can behave differently next month.

In [ ]:
!pip install numpy seaborn PyPortfolioOpt

## Imports and setup

risk_models.sample_cov estimates the covariance, BlackLittermanModel blends the views with the prior, plotting.plot_efficient_frontier draws the chart, and openbb pulls the closes, with NumPy doing the array math underneath.

In [ ]:
import numpy as np
import seaborn as sns

In [ ]:
from pypfopt import plotting, risk_models
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

In [ ]:
import yfinance as yf

Call it before the plot, because matplotlib reads the current rcParams when the figure is created, not when it's shown. Skip the line and the chart still draws, but it comes out on a white background with tighter fonts.

## Pull price history and state views

Pull daily closes for the five tickers with openbb.economy.index, which returns one column per symbol over the SDK's default date range.

In [ ]:
prices = yf.download(["AAPL", "BBY", "BAC", "SBUX", "T"], start="2020-01-01").Close

Check prices.count() first. The endpoint is named economy but it hands back equity closes, and if one ticker starts 600 rows later than the rest, pandas computes each covariance pair on only the dates those two share, so entries of the matrix come from different windows.

Write down what we expect each stock to return over the next year as a plain Python dictionary.

In [ ]:
viewdict = {
    "AAPL": 0.20,
    "BBY": 0.25,
    "BAC": 0.10,
    "SBUX": 0.2,
    "T": 0.15,
}

The 0.25 on BBY is a one-year total return forecast I typed by hand, not output from a valuation model, so treat all five as placeholders for your own work. Change it to 0.10, rerun, and watch how far the BBY weight moves.

## Blend views with market expectations

risk_models.sample_cov converts the closes to daily returns and annualizes the covariance by 252 trading days.

In [ ]:
cov_matrix = risk_models.sample_cov(prices)

The covariance matrix records how much each pair of stocks moves together, which is what separates a diversified bet from a concentrated one. Past average returns make poor forecasts, so the views supply the return estimate, and you can test how stable the co-movement is by splitting this sample in half and comparing the two matrices.

Blend the views with a prior return vector, using the library defaults of tau at 0.05 and view uncertainty built from the diagonal of tau times the covariance matrix.

In [ ]:
bl = BlackLittermanModel(
    cov_matrix,
    absolute_views=viewdict,
    pi="equal",
)

pi="equal" gives every stock the same prior expected return, one over the number of assets, instead of returns implied by market caps. absolute_views tells the model each number is a standalone expected return for one stock, so the view matrix is the identity across the five tickers.

## Build weights and plot frontier

Take the posterior returns from the model, hand them to EfficientFrontier for the chart, and get the allocation itself from bl_weights, which solves in closed form rather than through the frontier object.

In [ ]:
rets = bl.bl_returns()
ef = EfficientFrontier(rets, cov_matrix)
weights = bl.bl_weights()

weights comes back keyed by ticker, with anything under 0.0001 rounded to zero. BBY carries the highest view at 0.25, but it only outweighs T if its variance and its correlations with the other four don't eat the extra return, which you can check against each ticker's row in cov_matrix.

Plot the frontier with show_tickers=True, which draws the opportunity set and the five single stocks, not the bl_weights allocation.

In [ ]:
plotting.plot_efficient_frontier(ef, show_tickers=True)

The curve is the best return available at each risk level given the posterior returns and cov_matrix, and on this run the five ticker markers plot to the right of it, which is the diversification showing up. None of that says the views were right, and the only check is to hold the weights for the one-year horizon they were written for and compare the result against an equal-weight basket of the same five names.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.